In [ ]:
# This file is the main file for the WSPI PJK project

#=========== Imports =============#
# To apply math operations on arrays
import numpy as np

# To open and read image files
from PIL import Image

# Core ML library for building and training neural networks
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Tools built on torch specifically for image based ML tasks
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets # To load and label images based on folder structure

# To check file names
import os

import torchvision.transforms.functional as TF # To allow cropping and more image transformations

from torch.utils.data import random_split, DataLoader # To split up teh data and handle batching, shuffling, paralel loading, ect

In [17]:
# Check if being run on google colab or locally (Change paths as needed)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = '/content/drive/Shareddrives/WSPI/catergorized2'
    save_path = '/content/drive/MyDrive/trained_net.pth'
except ImportError:
    base_path = './data/images'
    save_path = './trained_net.pth'

#=== Constants ====#
image_size = 224

# Check if the current device has a NAVIDIA GPU to run things on, and use cpu if not
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using: cuda


In [18]:
#====== Use only if on google colab to download images to the enviorment (Faster) =========#
try:
    import shutil
    local_data = '/content/data'
    if not os.path.exists(local_data):
        print("Copying data to local disk...")
        shutil.copytree(base_path, local_data)
        print("Done!")
    base_path = local_data
except Exception:
    pass

Copying data to local disk...
Done!


In [ ]:
# Define how each image will be loaded and preprocessed before given to the model
class RoadDataset(datasets.ImageFolder):

    # Called whenever an image is accessed (ex. data[0])
    def __getitem__(self, index):
        # 1) Get the path, label, image size, and the image itself
        path, label_idx = self.samples[index]
        image = self.loader(path) # Loads as PIL (Python Imaging Library) Image
        image = image.convert('RGB') # Ensure the image only has RGB channels
        width, height = image.size

        # 2) Crop images to the center if they are above a certain size
        if width >= 400 and height >= 400:
          image = TF.center_crop(image, (400, 400))

        # 3) Apply transforms to the image (resize, tensor, normalize)
        if self.transform is not None:
            image = self.transform(image)

        # 4) Return the transformed image and index of label
        return (image, label_idx)

    # Get a list of class names
    def get_classes(self):
        return self.classes


In [21]:
# Define the transformations to be applied to each image
post_crop_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)), # Set standard image size
    transforms.ToTensor(), # Normalizes values to a range of 0 - 1 instead of 0-255 for the RGB channels of pixels

    # Centers RGB values around zero using ImageNet standard mean/std. Helps training converge(the model learns and stabilizes) faster
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create the dataset with the files and the transformations applied
full_dataset = RoadDataset(root=base_path, transform=post_crop_transform)

# Get class name array to assign based on indexes
class_names = full_dataset.get_classes()
print(class_names)

['dry', 'ice', 'mixed', 'snow']


In [22]:
# Decide split size for training (%90) and testing (%10)
train_size = int(0.9 * len(full_dataset))
test_size = len(full_dataset) - train_size

# Split randomly (using a seed for reproducibility)
train_subset, test_subset = random_split(
    full_dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

#=== Create training and testing dataloaders (DataLoaders wrap the datasets and handle feeding data to the model during training)
train_loader = DataLoader(
    train_subset,
    batch_size=32, # Amount of images to feed at a time (loss is averaged for each batch)
    shuffle=True, # Randomizes data order each run (epoch) so the model doesn't learn the patterns based on ordering
    num_workers=2, # How many next batches to load in the background with parallel threads while the model trains on the current batch
    pin_memory=True # Pre loads batches into GPU-ready memory which speeds up data loading. Only matters when using a GPU
)

test_loader = DataLoader(
    test_subset,
    batch_size=32,
    shuffle=False, # Set to false since order doesn't matter for testing
    num_workers=2,
    pin_memory=True
)

In [23]:
# Neural network class that defines the model itself, inherits from PyTorch's base module (gives built-in training, saving, and GPU support)
class NeuralNet(nn.Module):

    # All neural network layers are defined here
    def __init__(self):
        # Initialize PyTorch's base module before defining our own layers (required for PyTorch to track weights)
        super().__init__()

        #======= Define layers ============#
        # Flow: Conv layers find patterns -> pooling simplifies -> FC layers decide what class it is.

        # Convolutional layer: slides a filter across the image to detect patterns (edges, textures, etc.)
        # Pool layer: shrinks the image by keeping the max value from each pool_size x pool_size block of pixels
        # Fully connected (FC) layer: combines detected patterns to make a final class prediction

        conv1_out = 12
        conv2_out = 24
        kernel_size = 5
        pool_size = 2

        # Calculate final image size after conv and pool layers
        # kernel_size - 1 because the filter can't hang off the edge of the image (loses pixels on each side of the image )
        # Example:
        # Image:  [1] [2] [3] [4] [5] [6] [7]
        # Filter position 1:  [1  2  3  4  5]  <- on pixel 3
        # Filter position 2:     [2  3  4  5  6] <- on pixel 4
        # Filter position 3:        [3  4  5  6  7] <- on pixel 5, can't check 1, 2, 6, 7 without overhang
        after_conv1 = (image_size - (kernel_size - 1)) // pool_size  # ex. 224 -> 220 -> 110
        after_conv2 = (after_conv1 - (kernel_size - 1)) // pool_size  # ex. 110 -> 106 -> 53
        final_size = after_conv2

        fc1_size = 120
        fc2_size = 76
        flattened_size = final_size * final_size * conv2_out
        num_classes = len(class_names)

        self.conv1 = nn.Conv2d(3, conv1_out, kernel_size) # Originally 3 input channels (RGB), 12 feature maps, 5x5 grid (kernel_size)
        self.conv2 = nn.Conv2d(conv1_out, conv2_out, kernel_size) # Will take the 12 feature maps from conv1 and makes 24 in a 5x5 grid

        self.pool = nn.MaxPool2d(pool_size, pool_size) # Looks at every 2x2 block of pixels and keeps only the brightest one

        self.fc1 = nn.Linear(flattened_size, fc1_size) # Compress all pattern data into a smaller representation
        self.fc2 = nn.Linear(fc1_size, fc2_size)
        self.fc3 = nn.Linear(fc2_size, num_classes) # One output per road condition class, each is givemn a score and greatest one wins

    # Connects the layers in the order and way we want to run images through
    def forward(self, x):
        #== CONVOLUTIONAL
        x = self.pool(F.relu(self.conv1(x))) #F.relu sets any negative value to zero to increase class separation accuracy
        x = self.pool(F.relu(self.conv2(x)))

        #== FLATTEN
        x = torch.flatten(x, 1) # Flattens the maps of features into single rows of numbers

        #== NEURAL NETWORK
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x) # No ReLU since negative scores mean "definitely not this category" and are useful
        return x

In [ ]:
# Set to True to load a previously trained model, False to train from scratch
load_saved_model = False

# Create an instance of the model
net = NeuralNet()
net.to(device) # To use GPU or CPU

if load_saved_model:
    net.load_state_dict(torch.load(save_path, map_location=device)) # Load the saved weights into the model (map_location ensures it works on GPU or CPU regardless of where it was trained)
    print("Loaded saved model from", save_path)
else:
    print("Starting with a fresh model")

# Used to handle loss calculations after the model predicts scores for each category
loss_function = nn.CrossEntropyLoss()

# Used to adjust the model's weights to reduce loss
optimizer = optim.SGD(
    net.parameters(), # Gives the optimizer access to all the weights in the model
    lr = 0.001, # sets the learning rate, so how big each adjustment step is. (Too big and the model overshoots, too small and it takes forever to learn)
    momentum = 0.9 # Used to keep 90% of the previous directions of modifications, and mix in 10% of the new direction
)

Starting with a fresh model


In [ ]:
# start the training, each epoch is an iteration of training where every image is seen once
for epoch in range(30):
    print(f'Training epoch {epoch}...')

    # Used to see the average loss per epoch
    running_loss = 0

    # Run through the data in the defined batches
    for data in train_loader:
        # Unpack the images and labels of the batch, then load them on the avalible device
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        # set all gradients (weight modifications from previous run) to zero
        optimizer.zero_grad()

        # Give the images to the model to classify and give scores. Runs the forward method in the NeuralNet class for each image
        outputs = net(images)

        # Calculate loss by comparing the predictions to the actual labels
        loss = loss_function(outputs, labels)

        # Figure out how much each weight contributed to the error. Each weight gets a gradient that says what direction and amount to adjust it for the next training tests
        loss.backward()

        # Apply the decided gradient to each weight, scaled by the learning rate defined in the optimizer
        optimizer.step()

        running_loss += loss.item() # Add up current loss for the epoch

    print(f'Loss: {running_loss / len(train_loader):.4f}') # average loss accross all batches for the epoch


In [26]:
# Export the weights/parameters, the model
torch.save(net.state_dict(), save_path)

In [ ]:
#====== Evaluate accuracy on any dataset (Call where needed) ======#
def evaluate(loader, dataName=""):
    correct = 0
    total = 0

    # Create a list the size of the classes to fill with amounts of correct predictionsin each class
    class_correct = [0] * len(class_names)
    class_total = [0] * len(class_names)

    # switch from training mode to evaluation mode, affects how some layers behave
    net.eval()

    # Run the test images through the model and determine the accuracy it has after training
    with torch.no_grad(): # Tells PyTorch not to track gradients during this section
        for data in loader:
            # Run the test data through the model
            images, labels = data
            images, labels = images.to(device), labels.to(device) # load them on the avalible device
            outputs = net(images)

            # Get the highest score of each image
            _, predicted = torch.max(outputs, 1)

            # Get the total amount of images and the amount that are correct where the predicted value matches the expected label
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Go through each image and determine the total of each class and how many images we predicted correctly
            for i in range(len(labels)):
                label_idx = labels[i].item()
                class_total[label_idx] += 1
                if predicted[i] == labels[i]:
                    class_correct[label_idx] += 1

    # For each class diplay the amount of correct predictions and the accuracy of each
    print(f"\n{dataName} Results:")
    print(f"Overall: {correct}/{total}  Accuracy: {100 * correct / total:.1f}%")
    for i, name in enumerate(class_names):
        acc = 100 * class_correct[i] / class_total[i] if class_total[i] > 0 else 0
        print(f"  {name}: {class_correct[i]}/{class_total[i]}  Accuracy: {acc:.1f}%")

In [ ]:
#====== Test on training test split(10%) ======#
evaluate(test_loader, "Test Set")